In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from src.data.load import load_data

import joblib

RANDOM_STATE = 42
RAW_PATH = Path('/home/nikitos/MLSystem/data/raw/UCI_Credit_Card.csv')



/tmp/ipykernel_11032/2767273931.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [30]:
df = load_data(RAW_PATH)

In [31]:
bill_features = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
df['BILL_mean'] = df[bill_features].mean(axis=1)
df['BILL_std'] = df[bill_features].std(axis=1)
df['BILL_grow'] = (df['BILL_AMT6'] > df['BILL_AMT1']).astype(int)
df['BILL_trend'] = df['BILL_AMT6'] - df['BILL_AMT1']
df

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month,BILL_mean,BILL_std,BILL_grow,BILL_trend
0,20000.0,2,2,1,24,2,2,-1,-1,-2,...,689.0,0.0,0.0,0.0,0.0,1,1284.000000,1761.633219,0,-3913.0
1,120000.0,2,2,2,26,-1,2,0,0,0,...,1000.0,1000.0,1000.0,0.0,2000.0,1,2846.166667,637.967841,1,579.0
2,90000.0,2,2,2,34,0,0,0,0,0,...,1500.0,1000.0,1000.0,1000.0,5000.0,0,16942.166667,6064.518593,0,-13690.0
3,50000.0,2,2,1,37,0,0,0,0,0,...,2019.0,1200.0,1100.0,1069.0,1000.0,0,38555.666667,10565.793518,0,-17443.0
4,50000.0,1,2,1,57,-1,0,-1,0,0,...,36681.0,10000.0,9000.0,689.0,679.0,0,18223.166667,10668.590074,1,10514.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000.0,1,3,1,39,0,0,0,0,0,...,20000.0,5003.0,3047.0,5000.0,1000.0,0,120891.500000,86697.439530,0,-172968.0
29996,150000.0,1,3,2,43,-1,-1,-1,-1,0,...,3526.0,8998.0,129.0,0.0,0.0,0,3530.333333,3200.534247,0,-1683.0
29997,30000.0,1,2,2,37,4,3,2,-1,0,...,0.0,22000.0,4200.0,2000.0,3100.0,1,11749.333333,9354.149660,1,15792.0
29998,80000.0,1,3,1,41,1,-1,0,0,0,...,3409.0,1178.0,1926.0,52964.0,1804.0,1,44435.166667,32992.487323,1,50589.0


In [32]:
df = df.drop(columns=bill_features)
df

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month,BILL_mean,BILL_std,BILL_grow,BILL_trend
0,20000.0,2,2,1,24,2,2,-1,-1,-2,...,689.0,0.0,0.0,0.0,0.0,1,1284.000000,1761.633219,0,-3913.0
1,120000.0,2,2,2,26,-1,2,0,0,0,...,1000.0,1000.0,1000.0,0.0,2000.0,1,2846.166667,637.967841,1,579.0
2,90000.0,2,2,2,34,0,0,0,0,0,...,1500.0,1000.0,1000.0,1000.0,5000.0,0,16942.166667,6064.518593,0,-13690.0
3,50000.0,2,2,1,37,0,0,0,0,0,...,2019.0,1200.0,1100.0,1069.0,1000.0,0,38555.666667,10565.793518,0,-17443.0
4,50000.0,1,2,1,57,-1,0,-1,0,0,...,36681.0,10000.0,9000.0,689.0,679.0,0,18223.166667,10668.590074,1,10514.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000.0,1,3,1,39,0,0,0,0,0,...,20000.0,5003.0,3047.0,5000.0,1000.0,0,120891.500000,86697.439530,0,-172968.0
29996,150000.0,1,3,2,43,-1,-1,-1,-1,0,...,3526.0,8998.0,129.0,0.0,0.0,0,3530.333333,3200.534247,0,-1683.0
29997,30000.0,1,2,2,37,4,3,2,-1,0,...,0.0,22000.0,4200.0,2000.0,3100.0,1,11749.333333,9354.149660,1,15792.0
29998,80000.0,1,3,1,41,1,-1,0,0,0,...,3409.0,1178.0,1926.0,52964.0,1804.0,1,44435.166667,32992.487323,1,50589.0


In [33]:
pay_features = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
df['PAY_max'] = np.max(df[pay_features], axis=1)
df['has_delay'] = (df['PAY_max'] > 0).astype(int)

In [34]:
payamt_features = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
df['never_paid'] = (df[payamt_features].sum(axis=1) == 0).astype(int)
df[payamt_features] = np.log1p(df[payamt_features])

In [35]:
df['UTILIZATION'] = df['BILL_mean'] / df['LIMIT_BAL']
df['LIMIT_BAL_log'] = np.log1p(df['LIMIT_BAL'])
df = df.drop(columns=['LIMIT_BAL'])
df

,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,...,default.payment.next.month,BILL_mean,BILL_std,BILL_grow,BILL_trend,PAY_max,has_delay,never_paid,UTILIZATION,LIMIT_BAL_log
0,2,2,1,24,2,2,-1,-1,-2,-2,...,1,1284.000000,1761.633219,0,-3913.0,2,1,0,0.064200,9.903538
1,2,2,2,26,-1,2,0,0,0,2,...,1,2846.166667,637.967841,1,579.0,2,1,0,0.023718,11.695255
2,2,2,2,34,0,0,0,0,0,0,...,0,16942.166667,6064.518593,0,-13690.0,0,0,0,0.188246,11.407576
3,2,2,1,37,0,0,0,0,0,0,...,0,38555.666667,10565.793518,0,-17443.0,0,0,0,0.771113,10.819798
4,1,2,1,57,-1,0,-1,0,0,0,...,0,18223.166667,10668.590074,1,10514.0,0,0,0,0.364463,10.819798
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,1,3,1,39,0,0,0,0,0,0,...,0,120891.500000,86697.439530,0,-172968.0,0,0,0,0.549507,12.301387
29996,1,3,2,43,-1,-1,-1,-1,0,0,...,0,3530.333333,3200.534247,0,-1683.0,0,0,0,0.023536,11.918397
29997,1,2,2,37,4,3,2,-1,0,0,...,1,11749.333333,9354.149660,1,15792.0,4,1,0,0.391644,10.308986
29998,1,3,1,41,1,-1,0,0,0,-1,...,1,44435.166667,32992.487323,1,50589.0,1,1,0,0.555440,11.289794


In [38]:
print(df.isna().sum().sort_values(ascending=False))
print(np.isinf(df.select_dtypes(include=[np.number])).sum())
print(df['default.payment.next.month'].value_counts(normalize=True))
print(df.columns.tolist())

SEX                           0
EDUCATION                     0
MARRIAGE                      0
AGE                           0
PAY_AMT1                      0
PAY_AMT2                      0
PAY_AMT3                      0
PAY_AMT4                      0
PAY_AMT5                      0
PAY_AMT6                      0
default.payment.next.month    0
BILL_mean                     0
BILL_std                      0
BILL_grow                     0
BILL_trend                    0
PAY_max                       0
has_delay                     0
never_paid                    0
UTILIZATION                   0
LIMIT_BAL_log                 0
dtype: int64
SEX                           0
EDUCATION                     0
MARRIAGE                      0
AGE                           0
PAY_AMT1                      0
PAY_AMT2                      0
PAY_AMT3                      0
PAY_AMT4                      0
PAY_AMT5                      0
PAY_AMT6                      0
default.payment.next.month 

In [37]:
df = df.drop(columns=pay_features)

In [39]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = df.drop(columns=['default.payment.next.month']).dropna()
vif_data = pd.DataFrame({
    'feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
print(vif_data.sort_values('VIF', ascending=False))

          feature       VIF
14        PAY_max  4.409403
15      has_delay  4.398055
17    UTILIZATION  3.491855
10      BILL_mean  3.230634
18  LIMIT_BAL_log  2.796457
8        PAY_AMT5  1.781991
13     BILL_trend  1.728784
7        PAY_AMT4  1.710308
11       BILL_std  1.682883
6        PAY_AMT3  1.670836
5        PAY_AMT2  1.668790
9        PAY_AMT6  1.657409
16     never_paid  1.646181
4        PAY_AMT1  1.635316
12      BILL_grow  1.450642
3             AGE  1.273598
2        MARRIAGE  1.232645
1       EDUCATION  1.133381
0             SEX  1.031474


In [40]:
numeric_features = [
    'AGE',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3',
    'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
    'BILL_mean', 'BILL_std', 'BILL_grow', 'BILL_trend',
    'never_paid', 'UTILIZATION', 'LIMIT_BAL_log',
    'PAY_max', 'has_delay'
]

categorical_features = [
    'SEX',
    'EDUCATION',
    'MARRIAGE',
]

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

preprocessor

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['AGE', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3',
                                  'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
                                  'BILL_mean', 'BILL_std', 'BILL_grow',
                                  'BILL_trend', 'never_paid', 'UTILIZATION',
                                  'LIMIT_BAL_log', 'PAY_max', 'has_delay']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['SEX', 'EDUCATION', 'MARRIAGE'])])

In [41]:
from pathlib import Path

MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(preprocessor, MODELS_DIR / 'preprocessor.pkl')
print(f"Препроцессор сохранён в {MODELS_DIR / 'preprocessor.pkl'}")

Препроцессор сохранён в ../models/preprocessor.pkl
